# 🤖 Huấn Luyện PPO 9D Đối Xứng Lẻ (`0.5 * (π(obs) − π(−obs))`, `w_pos = 0.30`, 5s, PWM `0.60/0.40`)

**Cấu hình đồng bộ với Firmware `Project_self` (STM32L476RG):**
- **Chu kỳ điều khiển:** 200 Hz (`Ts = 5 ms`), Encoder cập nhật mỗi `20 ms` (50 Hz, 1400 CPR, bộ lọc vận tốc $N=10$).
- **Chuỗi truyền động:** Hệ số khuếch đại PPO `1.30x` → Bộ lọc PWM `0.60 × u_raw + 0.40 × u_prev` → Làm tròn số nguyên → Bù vùng chết mềm `80 PWM` (deadzone `|u| ≤ 2 → 0`).
- **Không gian quan sát 9D:** $[\theta_t, \dot{\theta}_t, x_t, v_t, \theta_{t-1}, \dot{\theta}_{t-1}, x_{t-1}, v_{t-1}, a_{t-1}]$.
- **Kiến trúc đối xứng lẻ chuẩn xác (`odd_symmetric = true`):**
  - Actor huấn luyện & suy luận theo đúng công thức đã kiểm chứng: $a(\text{obs}) = 0.5 \times (\pi(\text{obs}) - \pi(-\text{obs}))$ (giữ nguyên hai tầng ẩn và triệt tiêu hoàn toàn lệch tâm tại $\text{obs} = 0$).
  - Hàm thưởng trở về bản mốc: `r_hold_bonus = 0.0`, `w_pos = 0.30`, `max_episode_steps = 1000` (5s), `--recenter-prob 0.0`.
- **Mốc đối chiếu mới:** Bản đối xứng trên trọng số `best_policy_wpos_030.npz` cũ (`best_policy_wpos_030_sym.npz`: `24/98` ngã, `57/98` giữ vị trí, `21/24` cổng STM32 — chỉ thiếu 3 ca `7.4V +3° 0N`, `25s -5° 0N`, `0° -1.5N`).
- **Cổng an toàn STM32:** Bản thắng lưu tại `ab_winner_policy.npz` & `rl_sb3_stm32_ab_winner.h`. Chỉ ghi đè `rl_sb3_stm32.h` trong `Project_self` khi đạt cổng `real_robot` (`18/18 + 4/4 + 2/2`), đối chiếu chuỗi 9D Python↔C và đo thời gian vòng 200 Hz trên STM32 (do công thức đối xứng chạy 2 lượt suy luận).

---
### Bước 1: Lấy code mới nhất từ GitHub, cài đặt thư viện & định nghĩa `run_step` (dừng ngay khi lỗi)

> **Lưu ý:** `git reset --hard` giữ nguyên các checkpoint cũ trong `outputs/checkpoints/` nếu bạn đang chạy tiếp trên cùng phiên Colab.

In [ ]:
import os
import sys
import subprocess

def run_step(cmd, required_files=None):
    """Chạy lệnh đồng bộ và DỪNG NGAY nếu lệnh trả về lỗi hoặc thiếu file đầu ra bắt buộc."""
    print(f"\n>>> RUN: {' '.join(cmd)}", flush=True)
    proc = subprocess.run(cmd)
    if proc.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại với mã lỗi {proc.returncode}: {' '.join(cmd)}")
    if required_files:
        missing = [f for f in required_files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(f"Thiếu file kết quả sau bước chạy: {missing}")

if os.path.exists("configs"):
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
elif os.path.exists("self_blancing"):
    os.chdir("self_blancing")
    run_step(["git", "fetch", "origin"])
    run_step(["git", "reset", "--hard", "origin/p1-balance-robot"])
else:
    run_step(["git", "clone", "-b", "p1-balance-robot", "https://github.com/nguyenthanhsang2306/self_blancing.git"])
    os.chdir("self_blancing")

run_step([
    sys.executable, "-m", "pip", "install", "-q",
    "gymnasium", "stable-baselines3", "torch", "matplotlib", "pyyaml", "tensorboard"
])
print(f"[Ready] Thư mục làm việc hiện tại: {os.getcwd()}")
print(f"[Check] best_policy_wpos_030.npz có sẵn: {os.path.exists('outputs/checkpoints/best_policy_wpos_030.npz')}")
print(f"[Check] best_policy_wpos_030_sym.npz có sẵn: {os.path.exists('outputs/checkpoints/best_policy_wpos_030_sym.npz')}")

---
### Bước 2: Train bản 5 giây đối xứng lẻ (`--symmetric`, `r_hold_bonus = 0.0`, `w_pos = 0.30`, `--recenter-prob 0.0`)

- Dùng công thức đối xứng lẻ `0.5 * (π(obs) - π(-obs))` ngay trong quá trình PPO học.
- Trở về hàm thưởng mốc (`r_hold_bonus = 0.0` trong `configs/default.yaml`), giữ lọc PWM `0.60/0.40` và chọn lọc 2 bước (`--top-k 3` + `final` → 22 ca không nhiễu → Top 2 trên 98 ca).

In [ ]:
run_step(
    [
        sys.executable, "train.py",
        "--config", "configs/default.yaml",
        "--w-pos", "0.30",
        "--recenter-prob", "0.0",
        "--top-k", "3",
        "--symmetric",
        "--tag", "wpos_030_5s_sym",
    ],
    required_files=[
        "outputs/checkpoints/best_policy_wpos_030_5s_sym.npz",
        "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym.npz",
        "outputs/checkpoints/final_policy_wpos_030_5s_sym.npz",
        "outputs/checkpoints/ppo_final_model_wpos_030_5s_sym.zip",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym.h",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_runner_up.h",
        "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_final.h",
        "outputs/reports/eval_report_wpos_030_5s_sym.md",
    ],
)

---
### Bước 3: Đối chiếu Top 2 bản mới (`best` & `runner_up`) với bản đối xứng mốc `best_policy_wpos_030_sym.npz` (57/98 giữ VT, 24/98 ngã)

- Nếu chưa có `outputs/checkpoints/best_policy_wpos_030_sym.npz`, ô này sẽ tự tạo từ `best_policy_wpos_030.npz` (bật cờ `odd_symmetric=True`), hoặc mở hộp thoại để bạn upload `best_policy_wpos_030.npz`.
- Kết quả lưu vào `outputs/reports/sym_policy_comparison.md`.

In [ ]:
from src.models import NumpyMLPPolicy

ckpt_030_prev = "outputs/checkpoints/best_policy_wpos_030.npz"
ckpt_030_sym = "outputs/checkpoints/best_policy_wpos_030_sym.npz"
ckpt_sym_best = "outputs/checkpoints/best_policy_wpos_030_5s_sym.npz"
ckpt_sym_runner = "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym.npz"

if not os.path.exists(ckpt_030_sym) and not os.path.exists(ckpt_030_prev):
    print(f"[Notice] Chưa thấy {ckpt_030_prev} hay {ckpt_030_sym} trong Colab runtime.")
    try:
        from google.colab import files
        print("Hãy chọn file best_policy_wpos_030.npz (hoặc best_policy_wpos_030_sym.npz) để upload...")
        uploaded = files.upload()
        os.makedirs("outputs/checkpoints", exist_ok=True)
        for fname, data in uploaded.items():
            base_name = os.path.basename(fname)
            dst = os.path.join("outputs/checkpoints", base_name if base_name.endswith(".npz") else "best_policy_wpos_030.npz")
            with open(dst, "wb") as uf:
                uf.write(data)
            print(f"[Upload] Đã nạp {fname} -> {dst}")
    except Exception as e:
        print(f"[Warning] Không mở được upload ({e}).")

if not os.path.exists(ckpt_030_sym) and os.path.exists(ckpt_030_prev):
    pol_m = NumpyMLPPolicy(input_dim=9, hidden_dims=(64, 64), output_dim=1)
    pol_m.load_weights(ckpt_030_prev)
    pol_m.odd_symmetric = True
    pol_m.save_weights(ckpt_030_sym)
    print(f"[Baseline] Đã tạo bản mốc đối xứng từ {ckpt_030_prev} -> {ckpt_030_sym}")

missing_ckpts = [p for p in [ckpt_030_sym, ckpt_sym_best, ckpt_sym_runner] if not os.path.exists(p)]
if missing_ckpts:
    raise FileNotFoundError(f"Thiếu checkpoint để so sánh 3 bản: {missing_ckpts}")

run_step(
    [
        sys.executable, "inference.py",
        "--compare",
        ckpt_030_sym,
        ckpt_sym_best,
        ckpt_sym_runner,
        "--export-winner",
        "--report-out", "outputs/reports/sym_policy_comparison.md",
    ],
    required_files=[
        "outputs/checkpoints/ab_winner_policy.npz",
        "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
        "outputs/reports/sym_policy_comparison.md",
    ],
)

---
### Bước 4: Vẽ đồ thị kiểm tra cho Top 2 bản mới (`best` và `runner_up`)

In [ ]:
from IPython.display import Image, display

plot_jobs = [
    ("wpos_030_5s_sym #1 | Góc +5.0° chuẩn 10s", "outputs/checkpoints/best_policy_wpos_030_5s_sym.npz", ["--angle", "5.0", "--duration", "10.0"], "outputs/figures/wpos_030_5s_sym_best_angle5_10s.png"),
    ("wpos_030_5s_sym #1 | Góc -5.0° chạy dài 25s", "outputs/checkpoints/best_policy_wpos_030_5s_sym.npz", ["--angle", "-5.0", "--duration", "25.0"], "outputs/figures/wpos_030_5s_sym_best_neg5_drift25s.png"),
    ("wpos_030_5s_sym #1 | Góc 0.0° + cú đẩy -1.5 N (10s)", "outputs/checkpoints/best_policy_wpos_030_5s_sym.npz", ["--angle", "0.0", "--kick", "-1.5", "--duration", "10.0"], "outputs/figures/wpos_030_5s_sym_best_angle0_kickneg1p5N.png"),
    ("wpos_030_5s_sym #2 | Góc +5.0° chuẩn 10s", "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym.npz", ["--angle", "5.0", "--duration", "10.0"], "outputs/figures/wpos_030_5s_sym_runner_angle5_10s.png"),
    ("wpos_030_5s_sym #2 | Góc -5.0° chạy dài 25s", "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym.npz", ["--angle", "-5.0", "--duration", "25.0"], "outputs/figures/wpos_030_5s_sym_runner_neg5_drift25s.png"),
    ("wpos_030_5s_sym #2 | Góc 0.0° + cú đẩy -1.5 N (10s)", "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym.npz", ["--angle", "0.0", "--kick", "-1.5", "--duration", "10.0"], "outputs/figures/wpos_030_5s_sym_runner_angle0_kickneg1p5N.png"),
]

for title, ckpt, extra_args, fig_out in plot_jobs:
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"Dừng vẽ đồ thị vì thiếu checkpoint: {ckpt}")
    print(f"\n=== {title} ===", flush=True)
    run_step(
        [sys.executable, "inference.py", "--weights", ckpt, *extra_args, "--output-figure", fig_out],
        required_files=[fig_out],
    )
    display(Image(filename=fig_out))

---
### Bước 5: Kiểm tra sai số toán học & tính đối xứng xuất C Header (`PyTorch ↔ Numpy ↔ C`)

In [ ]:
run_step([sys.executable, "verify_export.py"])
run_step([
    sys.executable, "verify_export.py",
    "--weights", "outputs/checkpoints/final_policy_wpos_030_5s_sym.npz",
    "--sb3-model", "outputs/checkpoints/ppo_final_model_wpos_030_5s_sym.zip",
    "--header", "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_final.h",
])
run_step([
    sys.executable, "verify_export.py",
    "--weights", "outputs/checkpoints/ab_winner_policy.npz",
    "--header", "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
])

---
### Bước 6: Đóng gói `rl_sym_bundle.zip` & Tải về

In [ ]:
import zipfile

required_bundle_files = [
    "outputs/checkpoints/best_policy_wpos_030_sym.npz",
    "outputs/checkpoints/best_policy_wpos_030_5s_sym.npz",
    "outputs/checkpoints/runner_up_policy_wpos_030_5s_sym.npz",
    "outputs/checkpoints/final_policy_wpos_030_5s_sym.npz",
    "outputs/checkpoints/ab_winner_policy.npz",
    "configs/default.yaml",
    "outputs/reports/sym_policy_comparison.md",
    "outputs/reports/eval_report_wpos_030_5s_sym.md",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym.h",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_runner_up.h",
    "outputs/firmware_headers/rl_sb3_stm32_wpos_030_5s_sym_final.h",
    "outputs/firmware_headers/rl_sb3_stm32_ab_winner.h",
    "outputs/figures/wpos_030_5s_sym_best_angle5_10s.png",
    "outputs/figures/wpos_030_5s_sym_best_neg5_drift25s.png",
    "outputs/figures/wpos_030_5s_sym_best_angle0_kickneg1p5N.png",
    "outputs/figures/wpos_030_5s_sym_runner_angle5_10s.png",
    "outputs/figures/wpos_030_5s_sym_runner_neg5_drift25s.png",
    "outputs/figures/wpos_030_5s_sym_runner_angle0_kickneg1p5N.png",
]

missing_files = [p for p in required_bundle_files if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError(f"Dừng đóng gói vì thiếu file kết quả: {missing_files}")

optional_files = ["outputs/firmware_headers/rl_sb3_stm32.h"]
bundle_files = required_bundle_files + [p for p in optional_files if os.path.exists(p)]

zip_name = "rl_sym_bundle.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fpath in bundle_files:
        zf.write(fpath, arcname=fpath)
        print(f"  + {fpath}")

print(f"\n[Bundle] Đã đóng gói thành công {len(bundle_files)} file vào {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
    files.download("outputs/reports/sym_policy_comparison.md")
except Exception as e:
    print(f"Tự động tải bị chặn ({e}). Hãy tải thủ công từ cột trái Colab: {zip_name}")